# 🔬 Aggregate Detectie — Training Script v10 (Verbeterde Preprocessing)

**Verbeteringen t.o.v. v9:**
1. **Verbeterde preprocessing** — CLAHE vervangen door Gaussian background subtraction + multi-scale top-hat
2. **Enkelvoudig signaalkanaal** — model gebruikt alleen het echte fluorescentiekanaal (kanaal 0)
3. **Betere achtergrondonderdrukking** — Gaussian BG subtraction (σ=50) verwijdert diffuse achtergrond
4. **Multi-scale top-hat** — combineert kleine (r=4) en grote (r=8) schaal voor variabele aggregaatgrootten
5. **Alle v9 anti-overfitting maatregelen** blijven behouden

**Volgorde:**
1. Stap 1 — Installeer packages
2. Stap 2 — Verbind Google Drive
3. Stap 3 — Training starten
4. Stap 4 *(optioneel)* — Inferentie op nieuwe beelden

In [ ]:
# Stap 1: Installeer benodigde packages
!pip install segmentation_models_pytorch tifffile scikit-image scipy albumentations timm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 6.1 MB/s eta 0:00:00


In [ ]:
# Stap 2: Verbind Google Drive
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive verbonden!')

Mounted at /content/drive
✅ Google Drive verbonden!


## Stap 3 — Training

Pas `IMAGE_DIR`, `MASK_DIR` en `SAVE_DIR` aan naar jouw Google Drive mappen en voer de cel uit.

> **v10 preprocessing:** CLAHE is vervangen door Gaussian background subtraction (σ=50) gecombineerd met multi-scale top-hat filtering. Dit sluit aan bij de visuele pipeline die in het Protein Aggregate Analyzer programma is gevalideerd.

In [ ]:
# Stap 3: Training v10 — Verbeterde Preprocessing

import os, random, warnings
from pathlib import Path
import numpy as np
from PIL import Image
from scipy import ndimage
from skimage.morphology import disk, white_tophat
from skimage.filters import gaussian
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp

warnings.filterwarnings("ignore")

# ══════════════════════════════════════════════════════════════════════════════
# REPRODUCIBILITEIT
# ══════════════════════════════════════════════════════════════════════════════
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

set_seed(42)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# ══════════════════════════════════════════════════════════════════════════════
# 1. PREPROCESSING MET GAUSSIAN BG SUBTRACTION + MULTI-SCALE TOP-HAT (v10)
# ══════════════════════════════════════════════════════════════════════════════
# Redenering:
#   - Confocale beelden hebben bijna uitsluitend signaal op kanaal 0 (rood).
#   - CLAHE verhoogt lokaal contrast overal — ook in achtergrondgebieden —
#     waardoor het onderscheid tussen aggregaat en cytoplasma kleiner wordt.
#   - Gaussian background subtraction (σ=50) modelleert de diffuse, langzame
#     achtergrondvariatie en trekt die eraf. Aggregaten (kleine, scherpe spots)
#     overleven dit filter; diffuus cytoplasma wordt onderdrukt.
#   - Multi-scale top-hat combineert r=4 en r=8 zodat zowel kleine als iets
#     grotere aggregaten optimaal naar voren komen.
#   - Kanaal 3 = ruwe percentile-genormaliseerde intensiteit als context voor
#     het model (absolute helderheid).
# ══════════════════════════════════════════════════════════════════════════════
from scipy.ndimage import gaussian_filter as scipy_gaussian_filter

class Preprocessor:
    @staticmethod
    def pct_norm(img: np.ndarray) -> np.ndarray:
        lo, hi = np.percentile(img, [1.0, 99.9])
        if hi <= lo:
            return np.zeros_like(img, dtype=np.float32)
        return np.clip((img - lo) / (hi - lo), 0, 1).astype(np.float32)

    @staticmethod
    def build(img: np.ndarray) -> np.ndarray:
        # Gebruik altijd kanaal 0 — confocale beelden hebben hier het signaal
        raw  = img[:, :, 0].astype(np.float32) if img.ndim == 3 else img.astype(np.float32)
        norm = Preprocessor.pct_norm(raw)

        # Kanaal 1: Gaussian background subtraction (σ=50)
        # Trekt de diffuse achtergrond (cytoplasma, autofluorescentie) eraf.
        # σ=50 is veel groter dan een aggregaat maar kleiner dan een cellichaam.
        bg   = scipy_gaussian_filter(norm, sigma=50)
        ch1  = Preprocessor.pct_norm(np.clip(norm - bg, 0, None))

        # Kanaal 2: Multi-scale top-hat (r=4 en r=8 gecombineerd)
        # Top-hat filtert lokale intensiteitspieken die kleiner zijn dan de schijf.
        # Door twee schalen te combineren vangen we aggregaten van variabele grootte.
        th_small = white_tophat(norm, disk(4)).astype(np.float32)
        th_large = white_tophat(norm, disk(8)).astype(np.float32)
        ch2 = Preprocessor.pct_norm((th_small + th_large) / 2.0)

        # Kanaal 3: Ruwe percentile-genormaliseerde intensiteit (context)
        # Geeft het model toegang tot de absolute helderheid, zodat het kan
        # onderscheiden tussen echte aggregaten en artefacten.
        ch3 = norm

        stacked = np.stack([ch1, ch2, ch3], axis=0)
        mean = np.array([0.485, 0.456, 0.406], dtype=np.float32).reshape(3, 1, 1)
        std  = np.array([0.229, 0.224, 0.225], dtype=np.float32).reshape(3, 1, 1)
        return (stacked - mean) / std

# ══════════════════════════════════════════════════════════════════════════════
# 2. AUGMENTATIE & DATASET
# ══════════════════════════════════════════════════════════════════════════════
def elastic_transform(image, mask, alpha=200, sigma=12, seed=None):
    """Elastische vervorming — simuleert biologische variatie."""
    rng   = np.random.RandomState(seed)
    shape = image.shape[-2:]
    dx = gaussian(rng.randn(*shape) * alpha, sigma).astype(np.float32)
    dy = gaussian(rng.randn(*shape) * alpha, sigma).astype(np.float32)

    y, x   = np.meshgrid(np.arange(shape[0]), np.arange(shape[1]), indexing='ij')
    map_y  = np.clip(y + dy, 0, shape[0] - 1).astype(np.float32)
    map_x  = np.clip(x + dx, 0, shape[1] - 1).astype(np.float32)

    from scipy.ndimage import map_coordinates
    aug_img  = np.stack([
        map_coordinates(image[c], [map_y, map_x], order=1, mode='reflect')
        for c in range(image.shape[0])
    ], axis=0).astype(np.float32)
    aug_mask = map_coordinates(mask, [map_y, map_x], order=0, mode='reflect').astype(np.float32)
    return aug_img, aug_mask


def mixup(img1, msk1, img2, msk2, alpha=0.3):
    """
    Mixup augmentatie: mengt twee beelden en hun maskers lineair.
    Maakt de taak moeilijker en voorkomt overfitting op individuele beelden.
    alpha bepaalt de sterkte van de mix (klein alpha = zwakke mix).
    """
    lam = np.random.beta(alpha, alpha)
    lam = max(lam, 1 - lam)  # altijd >= 0.5, zodat het ene beeld dominant blijft
    mixed_img = lam * img1 + (1 - lam) * img2
    mixed_msk = lam * msk1 + (1 - lam) * msk2
    return mixed_img.astype(np.float32), mixed_msk.astype(np.float32)


class AggregateDataset(Dataset):
    def __init__(self, image_paths, mask_paths, tile=256, augment=True, repeat=20):
        self.paths   = list(zip(image_paths, mask_paths))
        self.tile    = tile
        self.augment = augment
        self.repeat  = repeat
        self.imgs, self.masks, self.pos_coords = [], [], []

        for ip, mp in self.paths:
            raw  = np.array(Image.open(str(ip))).astype(np.float32)
            mraw = np.array(Image.open(str(mp))).astype(np.float32)
            thr  = mraw.max() * 0.5 if mraw.max() > 1 else 0.5
            bin_ = ndimage.binary_fill_holes(mraw > thr).astype(np.float32)
            self.imgs.append(Preprocessor.build(raw))
            self.masks.append(bin_)
            self.pos_coords.append(np.argwhere(bin_ > 0.5))

    def __len__(self):
        return len(self.paths) * self.repeat

    def _get_tile(self, fi):
        """Snijdt een tile uit, met voorkeur voor positieve pixels."""
        proc = self.imgs[fi]
        mask = self.masks[fi]
        h, w = mask.shape
        ts   = self.tile

        if h < ts or w < ts:
            pad_h = max(0, ts - h)
            pad_w = max(0, ts - w)
            proc  = np.pad(proc, ((0,0), (0, pad_h), (0, pad_w)), mode='reflect')
            mask  = np.pad(mask, ((0, pad_h), (0, pad_w)), mode='reflect')
            h, w  = mask.shape

        coords = self.pos_coords[fi]
        if self.augment and len(coords) > 0 and random.random() < 0.7:
            cy, cx = coords[random.randint(0, len(coords)-1)]
            jr  = ts // 4
            y0  = int(np.clip(cy - ts//2 + random.randint(-jr, jr), 0, h-ts))
            x0  = int(np.clip(cx - ts//2 + random.randint(-jr, jr), 0, w-ts))
        else:
            y0 = random.randint(0, h - ts)
            x0 = random.randint(0, w - ts)

        return proc[:, y0:y0+ts, x0:x0+ts].copy(), mask[y0:y0+ts, x0:x0+ts].copy()

    def __getitem__(self, idx):
        fi       = idx % len(self.paths)
        p_img, p_msk = self._get_tile(fi)

        if self.augment:
            # Geometrische augmentaties
            if random.random() > 0.5:
                p_img = np.flip(p_img, axis=-1).copy()
                p_msk = np.flip(p_msk, axis=-1).copy()
            if random.random() > 0.5:
                p_img = np.flip(p_img, axis=-2).copy()
                p_msk = np.flip(p_msk, axis=-2).copy()
            k = random.randint(0, 3)
            if k:
                p_img = np.rot90(p_img, k, axes=(-2, -1)).copy()
                p_msk = np.rot90(p_msk, k).copy()

            # Elastische vervorming (25% kans)
            if random.random() < 0.25:
                alpha = random.uniform(100, 250)
                sigma = random.uniform(8, 15)
                p_img, p_msk = elastic_transform(p_img, p_msk, alpha=alpha, sigma=sigma)

            # Intensiteitsruis
            if random.random() < 0.4:
                noise_std = random.uniform(0.01, 0.04)
                p_img = p_img + np.random.randn(*p_img.shape).astype(np.float32) * noise_std

            # Contrast jitter
            if random.random() < 0.4:
                gamma = random.uniform(0.85, 1.15)
                p_img = np.sign(p_img) * (np.abs(p_img) ** gamma)

            # Cutout (20% kans — iets minder agressief dan v8)
            if random.random() < 0.2:
                cut_size = random.randint(16, 40)
                cy = random.randint(0, self.tile - cut_size)
                cx = random.randint(0, self.tile - cut_size)
                p_img[:, cy:cy+cut_size, cx:cx+cut_size] = 0.0

            # ── NIEUW: Mixup (20% kans) ─────────────────────────────────────
            # Meng met een willekeurig ander beeld uit de dataset.
            # Voorkomt dat het model specifieke trainingsbeelden memoriseert.
            if random.random() < 0.20 and len(self.paths) > 1:
                fi2 = random.choice([j for j in range(len(self.paths)) if j != fi])
                img2, msk2 = self._get_tile(fi2)
                p_img, p_msk = mixup(p_img, p_msk, img2, msk2, alpha=0.3)

        p_msk = (p_msk > 0.5).astype(np.float32)
        return torch.from_numpy(p_img.copy()), torch.from_numpy(p_msk[np.newaxis].copy())

# ══════════════════════════════════════════════════════════════════════════════
# 3. VERLIESFUNCTIE MET LABEL SMOOTHING
# ══════════════════════════════════════════════════════════════════════════════
class TverskyLoss(nn.Module):
    """
    Tversky Loss: penaliseert False Negatives (gemiste aggregaten) zwaarder.
    beta > alpha → hogere recall, minder gemiste aggregaten.
    """
    def __init__(self, alpha=0.3, beta=0.7, smooth=1e-6):
        super().__init__()
        self.alpha  = alpha
        self.beta   = beta
        self.smooth = smooth

    def forward(self, logits, target):
        p  = torch.sigmoid(logits)
        tp = (p * target).sum()
        fp = (p * (1 - target)).sum()
        fn = ((1 - p) * target).sum()
        tversky = (tp + self.smooth) / (tp + self.alpha * fp + self.beta * fn + self.smooth)
        return 1 - tversky


tversky_loss_fn = TverskyLoss(alpha=0.3, beta=0.7)

def robust_loss(logits, target, label_smoothing=0.05):
    """
    Gecombineerde loss met label smoothing (NIEUW in v9):
    - Label smoothing: vervangt harde 0/1 labels door 0.05/0.95.
      Voorkomt dat het model te zeker wordt en overfit op trainingsdata.
    - BCE met pos_weight=2.0 voor recall-boost.
    - Tversky met beta=0.7.
    Gewichten: 0.4 * BCE + 0.6 * Tversky.
    """
    # Label smoothing: zachte labels in plaats van harde 0/1
    smooth_target = target * (1 - label_smoothing) + 0.5 * label_smoothing

    weight = torch.tensor([2.0], device=logits.device)
    bce    = F.binary_cross_entropy_with_logits(logits, smooth_target, pos_weight=weight)
    tvr    = tversky_loss_fn(logits, target)  # Tversky op harde labels
    return 0.4 * bce + 0.6 * tvr


def compute_metrics(proba: torch.Tensor, target: torch.Tensor, threshold: float = 0.5) -> dict:
    p  = (proba > threshold).float().view(-1)
    t  = target.float().view(-1)
    tp = (p * t).sum().item()
    fp = (p * (1-t)).sum().item()
    fn = ((1-p) * t).sum().item()
    pr = tp / (tp + fp + 1e-8)
    rc = tp / (tp + fn + 1e-8)
    f1 = 2 * tp / (2 * tp + fp + fn + 1e-8)
    return {"f1": f1, "precision": pr, "recall": rc, "threshold": threshold}


def find_best_threshold(proba: torch.Tensor, target: torch.Tensor) -> dict:
    """Zoekt de beste threshold in stapjes van 0.025."""
    best = {"f1": 0.0, "precision": 0.0, "recall": 0.0, "threshold": 0.5}
    for thr in np.arange(0.25, 0.80, 0.025):
        m = compute_metrics(proba, target, threshold=float(thr))
        if m["f1"] > best["f1"]:
            best = m
    return best

# ══════════════════════════════════════════════════════════════════════════════
# 4. WARMUP + COSINE LEARNING RATE SCHEDULER
# ══════════════════════════════════════════════════════════════════════════════
class WarmupCosineScheduler:
    """
    Linear warmup voor de eerste `warmup_epochs` epochs,
    daarna cosine annealing naar eta_min.
    """
    def __init__(self, optimizer, warmup_epochs, total_epochs, eta_min=1e-6):
        self.optimizer     = optimizer
        self.warmup_epochs = warmup_epochs
        self.total_epochs  = total_epochs
        self.eta_min       = eta_min
        self.base_lrs      = [pg['lr'] for pg in optimizer.param_groups]
        self._epoch        = 0

    def step(self):
        self._epoch += 1
        e = self._epoch
        for pg, base_lr in zip(self.optimizer.param_groups, self.base_lrs):
            if e <= self.warmup_epochs:
                pg['lr'] = base_lr * (e / self.warmup_epochs)
            else:
                progress = (e - self.warmup_epochs) / (self.total_epochs - self.warmup_epochs)
                pg['lr'] = self.eta_min + 0.5 * (base_lr - self.eta_min) * (1 + np.cos(np.pi * progress))

    def get_last_lr(self):
        return [pg['lr'] for pg in self.optimizer.param_groups]

# ══════════════════════════════════════════════════════════════════════════════
# 5. TRAININGSLOOP
# ══════════════════════════════════════════════════════════════════════════════
def train_one_fold(trn_img, trn_msk, val_img, val_msk, save_path, fold_id, epochs=60):
    print(f"\n{'─'*75}")
    print(f"  FOLD {fold_id} — Val: {val_img[0].name}")
    print(f"{'─'*75}")

    trn_ds = AggregateDataset(trn_img, trn_msk, tile=256, augment=True,  repeat=50)
    # ── AANPASSING v9: val repeat omlaag (15→8) ──────────────────────────────
    # Bij kleine datasets bestaat de validatie anders uit veel overlappende
    # tiles van hetzelfde ene beeld, wat de val-loss onrealistisch stabiel maakt.
    val_ds = AggregateDataset(val_img, val_msk, tile=256, augment=False, repeat=8)

    trn_dl = DataLoader(trn_ds, batch_size=8, shuffle=True,  num_workers=2, pin_memory=True)
    val_dl = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=2, pin_memory=True)

    # ── Model: EfficientNet-B4 met dropout in decoder ───────────────────────────
    # `decoder_dropout` voegt dropout toe na elke decoder-laag.
    # Dit is de meest directe manier om overfitting in de decoder te remmen.
    model = smp.Unet(
        encoder_name="efficientnet-b4",
        encoder_weights="imagenet",
        in_channels=3,
        classes=1,
        decoder_attention_type="scse",
        decoder_dropout=0.3,           # ← NIEUW: 30% dropout in decoder
    ).to(DEVICE)

    encoder_params = list(model.encoder.parameters())
    decoder_params = list(model.decoder.parameters()) + list(model.segmentation_head.parameters())

    # ── AANPASSING v9: weight_decay verhoogd 1e-4 → 3e-4 ────────────────────
    # Sterkere L2-regularisatie remt grote gewichten en vermindert overfitting.
    optimizer = optim.AdamW([
        {'params': encoder_params, 'lr': 5e-5},
        {'params': decoder_params, 'lr': 2e-4},
    ], weight_decay=3e-4)

    scheduler = WarmupCosineScheduler(optimizer, warmup_epochs=5, total_epochs=epochs, eta_min=1e-6)

    best_f1   = 0.0
    best_thr  = 0.5
    patience  = 0
    MAX_PAT   = 20

    # ── NIEUW: EMA van recente thresholds voor stabielere threshold-keuze ────
    # In v8 sprong de threshold wild (0.25 – 0.78). Met een exponentieel
    # voortschrijdend gemiddelde (EMA) wordt de gekozen threshold stabieler.
    ema_thr   = 0.5
    EMA_ALPHA = 0.2   # hoe snel de EMA reageert op nieuwe waarden

    history = {'train_loss': [], 'val_loss': [], 'val_f1': [], 'val_recall': [], 'val_precision': []}

    print(f"  {'Ep':>4} | {'TrnL':>7} | {'ValL':>7} | {'F1':>6} | {'Rec':>6} | {'Prec':>6} | {'Thr':>5} | {'LR':>8}")
    print(f"  {'─'*87}")

    for epoch in range(1, epochs + 1):
        # --- TRAINING ---
        model.train()
        trn_loss = 0.0

        for imgs, masks in trn_dl:
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            optimizer.zero_grad()
            logits = model(imgs)
            loss   = robust_loss(logits, masks)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            trn_loss += loss.item()

        avg_trn = trn_loss / len(trn_dl)
        scheduler.step()
        current_lr = scheduler.get_last_lr()[1]

        # --- VALIDATIE ---
        model.eval()
        val_loss = 0.0
        all_preds, all_target = [], []

        with torch.no_grad():
            for imgs, masks in val_dl:
                imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
                logits = model(imgs)
                val_loss += robust_loss(logits, masks).item()
                all_preds.append(torch.sigmoid(logits).cpu())
                all_target.append(masks.cpu())

        avg_val = val_loss / len(val_dl)
        metrics = find_best_threshold(torch.cat(all_preds), torch.cat(all_target))

        # Update EMA threshold
        ema_thr = EMA_ALPHA * metrics["threshold"] + (1 - EMA_ALPHA) * ema_thr

        history['train_loss'].append(avg_trn)
        history['val_loss'].append(avg_val)
        history['val_f1'].append(metrics['f1'])
        history['val_recall'].append(metrics['recall'])
        history['val_precision'].append(metrics['precision'])

        print(f"  {epoch:>4d} | {avg_trn:>7.4f} | {avg_val:>7.4f} | "
              f"{metrics['f1']:>6.4f} | {metrics['recall']:>6.4f} | "
              f"{metrics['precision']:>6.4f} | {ema_thr:>5.2f} | {current_lr:>8.6f}", end="")

        if metrics["f1"] > best_f1:
            best_f1  = metrics["f1"]
            best_thr = round(ema_thr, 2)   # sla EMA-threshold op, niet de wilde per-epoch waarde
            patience = 0
            torch.save({
                "arch":       "unet",
                "encoder":    "efficientnet-b4",
                "state_dict": model.state_dict(),
                "tile":       256,
                "in_channels": 3,
                "best_f1":    best_f1,
                "best_thr":   best_thr,
                "version":    "v9",
            }, save_path)
            print("  ✓")
        else:
            patience += 1
            print()
            if patience >= MAX_PAT:
                print(f"\n  Early stop: {MAX_PAT} epochs geen verbetering.")
                break

    # --- PLOT ---
    plot_path = save_path.replace('.pth', '_learning_curve.png')
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    axes[0].plot(history['train_loss'], label='Train Loss', color='blue',   linewidth=2)
    axes[0].plot(history['val_loss'],   label='Val Loss',   color='orange', linewidth=2)
    axes[0].set_title(f'Loss (Fold {fold_id})'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    axes[0].legend(); axes[0].grid(True, linestyle='--', alpha=0.6)

    axes[1].plot(history['val_f1'], label='Val F1', color='green', linewidth=2)
    axes[1].set_title(f'F1 Score (Fold {fold_id})'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('F1')
    axes[1].legend(); axes[1].grid(True, linestyle='--', alpha=0.6)

    axes[2].plot(history['val_recall'],    label='Recall',    color='red',    linewidth=2)
    axes[2].plot(history['val_precision'], label='Precision', color='purple', linewidth=2)
    axes[2].set_title(f'Recall & Precision (Fold {fold_id})'); axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Score')
    axes[2].legend(); axes[2].grid(True, linestyle='--', alpha=0.6)

    plt.tight_layout()
    plt.savefig(plot_path, dpi=120)
    plt.close()

    print(f"\n  Fold {fold_id} klaar — beste F1: {best_f1:.4f} (EMA-Threshold: {best_thr:.2f})")
    print(f"  Grafiek opgeslagen: {Path(plot_path).name}")
    return best_f1

# ══════════════════════════════════════════════════════════════════════════════
# 6. LEAVE-ONE-OUT TRAINING LOOP
# ══════════════════════════════════════════════════════════════════════════════
def train_loo(image_dir, mask_dir, save_dir, epochs=60):
    os.makedirs(save_dir, exist_ok=True)
    image_paths = sorted(Path(image_dir).glob("*.tif*"))
    mask_paths  = sorted(Path(mask_dir).glob("*.tif*"))

    N = len(image_paths)
    print(f"\n{'#'*75}\n# START TRAINING v9 ({N} beelden gevonden)\n{'#'*75}")
    print(f"  Encoder:       EfficientNet-B4 (pretrained ImageNet)")
    print(f"  Preprocessing: Gaussian BG subtraction (σ=50) + multi-scale top-hat (r=4,8)")
    print(f"  Loss:          0.4 * BCE(label_smooth=0.05) + 0.6 * Tversky(α=0.3, β=0.7)")
    print(f"  Regularisatie: decoder_dropout=0.3 | weight_decay=3e-4")
    print(f"  Augmentatie:   flip + rot + elastisch + ruis + contrast + cutout + mixup")
    print(f"  LR schedule:   5 epochs warmup + cosine decay")
    print(f"  Threshold:     EMA-gestabiliseerd (α=0.2)")
    print(f"  Device:        {DEVICE}")

    all_f1 = []
    for fold_id in range(1, N + 1):
        val_i   = fold_id - 1
        trn_idx = [j for j in range(N) if j != val_i]

        trn_img = [image_paths[j] for j in trn_idx]
        trn_msk = [mask_paths[j]  for j in trn_idx]
        val_img = [image_paths[val_i]]
        val_msk = [mask_paths[val_i]]

        save_path = os.path.join(save_dir, f"model_fold{fold_id:02d}.pth")
        f1 = train_one_fold(trn_img, trn_msk, val_img, val_msk, save_path, fold_id, epochs=epochs)
        all_f1.append(f1)

    print(f"\n{'═'*75}")
    print(f"  Gemiddelde F1 over alle folds: {np.mean(all_f1):.4f}")
    print(f"  Per fold: {[f'{f:.4f}' for f in all_f1]}")
    print(f"{'═'*75}")

# ══════════════════════════════════════════════════════════════════════════════
# 7. PADEN & EXECUTION
# ══════════════════════════════════════════════════════════════════════════════
# ⚙️  Pas de paden hieronder aan naar jouw Google Drive mappen:
IMAGE_DIR = "/content/drive/MyDrive/Images"                  # map met .tif trainingsbeelden
MASK_DIR  = "/content/drive/MyDrive/Masks"                   # map met bijbehorende .tif maskers
SAVE_DIR  = "/content/drive/MyDrive/Modellen_v10_PREPROCESSING"   # map waar modellen worden opgeslagen

# ▶️  Start de training:
train_loo(IMAGE_DIR, MASK_DIR, SAVE_DIR, epochs=60)


Device: cuda

###########################################################################
# START TRAINING v9 (10 beelden gevonden)
###########################################################################
  Encoder:       EfficientNet-B4 (pretrained ImageNet)
  Preprocessing: Gaussian BG subtraction (σ=50) + multi-scale top-hat (r=4,8)
  Loss:          0.4 * BCE(label_smooth=0.05) + 0.6 * Tversky(α=0.3, β=0.7)
  Regularisatie: decoder_dropout=0.3 | weight_decay=3e-4
  Augmentatie:   flip + rot + elastisch + ruis + contrast + cutout + mixup
  LR schedule:   5 epochs warmup + cosine decay
  Threshold:     EMA-gestabiliseerd (α=0.2)
  Device:        cuda

───────────────────────────────────────────────────────────────────────────
  FOLD 1 — Val: Image_10_zonder_annotaties_1024.tiff
───────────────────────────────────────────────────────────────────────────


config.json:   0%|          | 0.00/106 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/77.9M [00:00<?, ?B/s]

    Ep |    TrnL |    ValL |     F1 |    Rec |   Prec |   Thr |       LR
  ───────────────────────────────────────────────────────────────────────────────────────
     1 |  0.6943 |  0.6185 | 0.5414 | 0.5246 | 0.5592 |  0.49 | 0.000040  ✓
     2 |  0.6128 |  0.6032 | 0.5811 | 0.5529 | 0.6123 |  0.49 | 0.000080  ✓
     3 |  0.5685 |  0.5037 | 0.6298 | 0.6432 | 0.6171 |  0.47 | 0.000120  ✓
     4 |  0.5055 |  0.5141 | 0.6242 | 0.6121 | 0.6368 |  0.46 | 0.000160
     5 |  0.4371 |  0.4746 | 0.6357 | 0.6227 | 0.6493 |  0.46 | 0.000200  ✓
     6 |  0.3856 |  0.4262 | 0.6741 | 0.6557 | 0.6936 |  0.41 | 0.000200  ✓
     7 |  0.3532 |  0.3743 | 0.6846 | 0.7004 | 0.6695 |  0.38 | 0.000199  ✓
     8 |  0.3436 |  0.6367 | 0.7282 | 0.7042 | 0.7538 |  0.39 | 0.000199  ✓
     9 |  0.3388 |  0.3658 | 0.6841 | 0.7023 | 0.6669 |  0.39 | 0.000197
    10 |  0.3270 |  0.3493 | 0.7128 | 0.7015 | 0.7245 |  0.38 | 0.000196
    11 |  0.3155 |  0.3161 | 0.7445 | 0.7363 | 0.7529 |  0.41 | 0.000194  ✓
    12 |  

## Stap 4 (optioneel) — Inferentie op een nieuw beeld

Pas `NIEUW_BEELD_PAD` aan naar het pad van jouw `.tiff` bestand.

In [ ]:
# Stap 4: Inferentie op nieuwe beelden
import tifffile

# ══════════════════════════════════════════════════════════════════════════════
# INFERENTIE — ENSEMBLE OVER ALLE OPGELEVERDE MODELLEN
# ══════════════════════════════════════════════════════════════════════════════
def predict_tta(model, processed, tile=256, overlap=64):  # overlap verhoogd van 32 → 64
    model.eval()
    _, h, w = processed.shape
    stride = tile - overlap
    pred_sum  = np.zeros((h, w), dtype=np.float32)
    count_map = np.zeros((h, w), dtype=np.float32)

    hann   = np.hanning(tile + 2)[1:-1]
    window = np.outer(hann, hann).astype(np.float32)

    ys = list(range(0, max(1, h - tile + 1), stride))
    xs = list(range(0, max(1, w - tile + 1), stride))
    if not ys or ys[-1] + tile < h: ys.append(max(0, h - tile))
    if not xs or xs[-1] + tile < w: xs.append(max(0, w - tile))

    # 8-voudige TTA: 4 rotaties × 2 spiegelingen
    tta = [(False,0),(False,1),(False,2),(False,3),(True,0),(True,1),(True,2),(True,3)]

    for y0 in ys:
        for x0 in xs:
            y1, x1 = min(y0+tile, h), min(x0+tile, w)
            patch = processed[:, y0:y1, x0:x1]
            ph, pw = y1-y0, x1-x0

            if ph < tile or pw < tile:
                pad = np.zeros((3, tile, tile), dtype=np.float32)
                pad[:, :ph, :pw] = patch
                patch = pad

            tta_p = []
            for flip, k in tta:
                aug = patch.copy()
                if flip: aug = np.flip(aug, axis=-1)
                if k:    aug = np.rot90(aug, k, axes=(-2,-1))

                t_ = torch.from_numpy(aug.copy()[np.newaxis]).float().to(DEVICE)
                with torch.no_grad():
                    p_ = torch.sigmoid(model(t_))[0,0].cpu().numpy()

                if k:    p_ = np.rot90(p_, -k)
                if flip: p_ = np.flip(p_, axis=-1)
                tta_p.append(p_)

            mp = np.mean(tta_p, axis=0)
            wp = window[:y1-y0, :x1-x0]
            pred_sum[y0:y1, x0:x1]  += mp[:y1-y0, :x1-x0] * wp
            count_map[y0:y1, x0:x1] += wp

    return np.where(count_map > 0, pred_sum / (count_map + 1e-8), 0)


def predict_new_image(image_path, model_dir):
    model_files = sorted(Path(model_dir).glob("model_fold*.pth"))
    if not model_files:
        raise FileNotFoundError(f"Geen modellen gevonden in {model_dir}!")

    print(f"\nInferentie: {Path(image_path).name}")
    print(f"Laadt {len(model_files)} modellen voor ensemble...")

    img_raw   = np.array(Image.open(str(image_path))).astype(np.float32)
    processed = Preprocessor.build(img_raw)

    all_proba, all_thr = [], []

    for mf in model_files:
        ckpt    = torch.load(str(mf), map_location=DEVICE, weights_only=False)
        version = ckpt.get("version", "v7")
        encoder = ckpt.get("encoder", "resnet34")

        model = smp.Unet(
            encoder_name=encoder,
            encoder_weights=None,
            in_channels=3,
            classes=1,
            decoder_attention_type="scse" if version in ("v8", "v9", "v10") else None
        ).to(DEVICE)
        model.load_state_dict(ckpt["state_dict"])

        tile_size = ckpt.get("tile", 256)
        opt_thr   = ckpt.get("best_thr", 0.5)

        proba = predict_tta(model, processed, tile=tile_size, overlap=64)
        all_proba.append(proba)
        all_thr.append(opt_thr)
        print(f"  ✓ {mf.name} [{encoder}] (threshold: {opt_thr:.2f})")

    ensemble_proba = np.mean(all_proba, axis=0)
    avg_threshold  = np.mean(all_thr)
    binary_mask    = (ensemble_proba > avg_threshold).astype(np.uint8) * 255
    n_agg          = ndimage.label(binary_mask > 0)[1]

    print(f"\nResultaat:")
    print(f"  Gemiddelde threshold:         {avg_threshold:.2f}")
    print(f"  Voorspelde aggregaten:        {n_agg}")

    return binary_mask, ensemble_proba


import tifffile

NIEUW_BEELD_PAD = "/content/drive/MyDrive/Nieuwe_beelden/jouw_beeld.tiff"
SAVE_DIR        = "/content/drive/MyDrive/Modellen_v9_ANTIOVERFITTING"

binary, proba = predict_new_image(NIEUW_BEELD_PAD, SAVE_DIR)

Image.fromarray(binary).save("voorspelling_binair.tif")
tifffile.imwrite("voorspelling_kansen.tif", (proba * 65535).astype(np.uint16))